In [1]:
%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from loguru import logger
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.generation import RagTemplate
from neo4j_graphrag.generation.graphrag import GraphRAG
from neo4j_graphrag.generation.prompts import PromptTemplate
from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.retrievers import HybridRetriever
from neo4j_graphrag.types import LLMMessage
from openai import OpenAI

from core import chat
from core.models import Settings

VECTOR_INDEX = "coins_embeddings_index"
FULL_TEXT_INDEX = "coins_text_index"
settings = Settings()


def find_coins_by_imperor(name: str) -> list[str]:
    driver = GraphDatabase.driver(
        settings.neo4j_uri,
        auth=(settings.neo4j_username, settings.neo4j_password),
    )
    with driver.session() as session:
        result = session.run(
            """
            MATCH (c:Coin)-[:HAS_AUTHORITY]->(e:Authority {name: $name})
            RETURN c LIMIT 10
            """,
            name=name,
        )

        data = [{
            "record_id": record["c"]["record_id"],
            "title": record["c"]["title"],
            "date_range": record["c"]["date_range"],
            "from_year": record["c"]["from_year"],
            "to_year": record["c"]["to_year"],
            "object_type": record["c"]["object_type"],
            "description": record["c"]["description"],
        } for record in result]
    driver.close()
    return data


coins = find_coins_by_imperor("Caracalla")
coins

[{'record_id': 'ric.4.crl.30a',
  'name': <Node element_id='4:ad1e36ad-9e05-455c-a915-4dedff0761f3:74' labels=frozenset({'Coin'}) properties={'date_range': '', 'record_id': 'ric.4.crl.30a', 'to_year': 200, 'from_year': 200, 'object_type': 'Coin', 'embedding': [0.008650551550090313, 0.028934603556990623, 0.036880094558000565, 0.0013211412588134408, -0.02855495549738407, -0.04471711441874504, -0.03270396590232849, -0.01739601418375969, 0.024297473952174187, -0.014304595068097115, 0.025490652769804, -0.013375813141465187, 0.005342190153896809, 0.05054742470383644, 0.06399781256914139, -0.06725193560123444, 0.011280969716608524, 0.024148326367139816, -0.015429980121552944, 0.07034335285425186, 0.009742039255797863, 0.01639265939593315, -0.0010838612215593457, -0.009714921936392784, -0.026670273393392563, -0.006325207185000181, -0.002691433299332857, -0.0061896187253296375, 0.0032320928294211626, -0.05776073411107063, 0.00511507922783494, -0.013979182578623295, -0.0009059011936187744, -0.04